# Phase 4: LLM evaluation with Ollama (gold vs retrieved contexts, LLM-as-judge)

In [1]:
# Setup: clone the repo and copy the datasets (same layout as retrieval-benchmark-full).
# The Hugging Face token comes from Kaggle Secrets (Add-ons > Secrets > HF_TOKEN); never paste it here.
!pip install -q "datasets<4.0.0" qdrant_client
!git clone https://github.com/Scrowbin/science-document-viewer
!cd science-document-viewer && git checkout feat/app-improvements && git log --oneline -1
!pip install -q -r science-document-viewer/backend/requirements.txt
import os
!apt install zstd
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["DEBUG"] = "True"  # backend settings need DEBUG or a SECRET_KEY
# os.environ["GEMINI_API"] = UserSecretsClient().get_secret("GEMINI_API")  # LLM judge (Gemini API)
!cp -r /kaggle/input/datasets/kristiko/qasper-official /kaggle/working/science-document-viewer/eval/data/
!mv /kaggle/working/science-document-viewer/eval/data/qasper-official /kaggle/working/science-document-viewer/eval/data/qasper_official
!mkdir -p /kaggle/working/science-document-viewer/eval/pdfs
!cp -r /kaggle/input/datasets/kristiko/peerqa /kaggle/working/science-document-viewer/eval/pdfs/peerqa
!find /kaggle/working/science-document-viewer/eval/pdfs/peerqa -name "*.pdf" | wc -l   # expect about 107

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
Cloning into 'science-document-viewer'...
remote: Enumerating objects: 2916, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 2916 (delta 11), reused 30 (delta 6), pack-reused 2858 (from 1)
Receiving objects:

In [2]:
# Install Ollama, then start TWO separate instances - one per GPU on separate ports.
# GPU 0 -> port 11434 (default), GPU 1 -> port 11435.
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, os

# Server on GPU 0 (port 11434)
subprocess.Popen(
    ["ollama", "serve"],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES="0", OLLAMA_HOST="127.0.0.1:11434"),
    stdout=open("/kaggle/working/ollama_gpu0.log", "w"), stderr=subprocess.STDOUT
)

# Server on GPU 1 (port 11435)
subprocess.Popen(
    ["ollama", "serve"],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES="1", OLLAMA_HOST="127.0.0.1:11435"),
    stdout=open("/kaggle/working/ollama_gpu1.log", "w"), stderr=subprocess.STDOUT
)

time.sleep(8)  # wait for both servers to start

# Verify both ports are up
import requests
for port, label in [(11434, 'GPU 0'), (11435, 'GPU 1')]:
    try:
        r = requests.get(f'http://127.0.0.1:{port}/api/tags', timeout=5)
        print(f'Ollama {label} :{port} OK')
    except Exception as e:
        print(f'Ollama {label} :{port} FAILED: {e}')

# Pull all models to both instances
MODELS = ["llama3.2:3b", "qwen2.5:1.5b", "phi4-mini", "qwen3:4b", "gemma3:4b"]
for m in MODELS:
    for port in (11434, 11435):
        r = requests.post(
            f'http://127.0.0.1:{port}/api/pull',
            json={'model': m, 'stream': False}, timeout=3600
        )
        r.raise_for_status()
    print(f'{m}: pulled to both GPUs')


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama GPU 0 :11434 OK
Ollama GPU 1 :11435 OK
llama3.2:3b: pulled to both GPUs
qwen2.5:1.5b: pulled to both GPUs
phi4-mini: pulled to both GPUs
qwen3:4b: pulled to both GPUs
gemma3:4b: pulled to both GPUs


In [3]:
J = 'gemini-flash-latest'  # active Gemini model (or local Ollama model)
S = 'science-document-viewer/eval/phases/phase_4_generation/scripts/evaluate_llm_generation.py'

In [4]:
import subprocess, os

p1 = subprocess.Popen(
    ['python', S, '--cohort', 'C-RET', '--condition', 'gold'],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES='0', OLLAMA_BASE_URL='http://127.0.0.1:11434'),
    stdout=open('/kaggle/working/gen_gold_cret.log', 'w'), stderr=subprocess.STDOUT
)
p2 = subprocess.Popen(
    ['python', S, '--cohort', 'PEERQA-PDF', '--condition', 'gold'],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES='1', OLLAMA_BASE_URL='http://127.0.0.1:11435'),
    stdout=open('/kaggle/working/gen_gold_peerqa.log', 'w'), stderr=subprocess.STDOUT
)

try:
    rc1 = p1.wait()
    rc2 = p2.wait()
finally:
    for name, log, rc in [('C-RET', '/kaggle/working/gen_gold_cret.log', rc1 if 'rc1' in dir() else '?'),
                           ('PEERQA-PDF', '/kaggle/working/gen_gold_peerqa.log', rc2 if 'rc2' in dir() else '?')]:
        lines = open(log).readlines()
        print(f'\n=== {name} (exit {rc}) last 20 lines ===')
        print(''.join(lines[-20:]))
    !cd science-document-viewer && zip -qr /kaggle/working/phase4_checkpoint.zip eval/phases/phase_4_generation/artifacts
    print('Checkpoint saved.')



=== C-RET (exit 0) last 20 lines ===
C-RET: 0 questions with references (skipped {'question_not_in_qasper_json': 567})
  llama3.2:3b: complete
  qwen2.5:1.5b: complete
  phi4-mini: complete
  qwen3:4b: complete
  gemma3:4b: complete


=== PEERQA-PDF (exit 0) last 20 lines ===
    100/181
    125/181
    150/181
    175/181
  qwen3:4b: 181 questions [PEERQA-PDF / gold]
    25/181
    50/181
    75/181
    100/181
    125/181
    150/181
    175/181
  gemma3:4b: 181 questions [PEERQA-PDF / gold]
    25/181
    50/181
    75/181
    100/181
    125/181
    150/181
    175/181

Checkpoint saved.


3) Retrieved contexts. Set PIPELINE to the arm chosen from Phase 3 (format `embedder:arm`, e.g. `qwen:convex+bgem3`, `e5:dense`).

In [5]:
PIPELINE = 'gemma:dense'  # <- Phase 3 choice
import subprocess, os

p1 = subprocess.Popen(
    ['python', S, '--cohort', 'C-RET', '--condition', 'retrieved', '--pipeline', PIPELINE],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES='0', OLLAMA_BASE_URL='http://127.0.0.1:11434'),
    stdout=open('/kaggle/working/gen_ret_cret.log', 'w'), stderr=subprocess.STDOUT
)
p2 = subprocess.Popen(
    ['python', S, '--cohort', 'PEERQA-PDF', '--condition', 'retrieved', '--pipeline', PIPELINE],
    env=dict(os.environ, CUDA_VISIBLE_DEVICES='1', OLLAMA_BASE_URL='http://127.0.0.1:11435'),
    stdout=open('/kaggle/working/gen_ret_peerqa.log', 'w'), stderr=subprocess.STDOUT
)

try:
    rc1 = p1.wait()
    rc2 = p2.wait()
finally:
    for name, log, rc in [('C-RET', '/kaggle/working/gen_ret_cret.log', rc1 if 'rc1' in dir() else '?'),
                           ('PEERQA-PDF', '/kaggle/working/gen_ret_peerqa.log', rc2 if 'rc2' in dir() else '?')]:
        lines = open(log).readlines()
        print(f'\n=== {name} (exit {rc}) last 20 lines ===')
        print(''.join(lines[-20:]))
    !cd science-document-viewer && zip -qr /kaggle/working/phase4_checkpoint.zip eval/phases/phase_4_generation/artifacts
    print('Checkpoint saved.')



=== C-RET (exit 0) last 20 lines ===
C-RET: 0 questions with references (skipped {'question_not_in_qasper_json': 567})
  llama3.2:3b: complete
  qwen2.5:1.5b: complete
  phi4-mini: complete
  qwen3:4b: complete
  gemma3:4b: complete


=== PEERQA-PDF (exit 0) last 20 lines ===
    100/181
    125/181
    150/181
    175/181
  qwen3:4b: 181 questions [PEERQA-PDF / retrieved_gemma_dense]
    25/181
    50/181
    75/181
    100/181
    125/181
    150/181
    175/181
  gemma3:4b: 181 questions [PEERQA-PDF / retrieved_gemma_dense]
    25/181
    50/181
    75/181
    100/181
    125/181
    150/181
    175/181

Checkpoint saved.


In [6]:
# Zip the results and download them from the Output tab, then unzip into the repo at the same path.
!cd science-document-viewer && zip -qr /kaggle/working/phase4_artifacts.zip eval/phases/phase_4_generation/artifacts